In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import torch
from torch.utils.data import DataLoader
import time

from src.data_loader import (load_ratings, train_test_split_by_time,
                              build_interaction_matrix)
from src.metrics import evaluate_rankings
from src.utils import set_seed
from src.models.ncf import (GMF, MLP, NeuMF, NCFDataset,
                              build_negative_samples, train_ncf_model,
                              predict_topk_ncf)

In [ ]:
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

ratings = load_ratings('../data/raw')
train, test = train_test_split_by_time(ratings, test_ratio=0.2)

N_USERS = 943
N_ITEMS = 1682
K = 10
THRESHOLD = 4

In [ ]:
# ground truth
test_gt = {}
for uid, group in test.groupby('user_id'):
    pos = group[group['rating'] >= THRESHOLD]['item_id'].tolist()
    if pos:
        test_gt[uid] = pos

train_user_items = {}
for uid, group in train.groupby('user_id'):
    train_user_items[uid] = set(group['item_id'].values)

print(f'{len(test_gt)} users to evaluate')

In [ ]:
# build training data with negative sampling
users_arr, items_arr, labels_arr = build_negative_samples(
    train, n_items=N_ITEMS, neg_ratio=4, seed=42)
print(f'training samples: {len(labels_arr)} (pos: {int(labels_arr.sum())}, neg: {int(len(labels_arr) - labels_arr.sum())})')

dataset = NCFDataset(users_arr, items_arr, labels_arr)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

## GMF only

In [ ]:
gmf = GMF(N_USERS, N_ITEMS, embed_dim=32)
gmf = train_ncf_model(gmf, loader, epochs=10, lr=0.001, device=device)

In [ ]:
all_items = list(range(1, N_ITEMS + 1))

def eval_model(model, tag=''):
    recs = {}
    for uid in test_gt:
        exclude = train_user_items.get(uid, set())
        recs[uid] = predict_topk_ncf(model, uid, all_items, exclude, k=K, device=device)
    metrics = evaluate_rankings(recs, test_gt, K)
    print(f'{tag}: {metrics}')
    return metrics

gmf_metrics = eval_model(gmf, 'GMF')

## MLP only

In [ ]:
mlp = MLP(N_USERS, N_ITEMS, embed_dim=32, hidden_layers=[64, 32, 16])
mlp = train_ncf_model(mlp, loader, epochs=10, lr=0.001, device=device)
mlp_metrics = eval_model(mlp, 'MLP')

## NeuMF (GMF + MLP)

In [ ]:
neumf = NeuMF(N_USERS, N_ITEMS, gmf_dim=32, mlp_dim=32, mlp_hidden=[64, 32, 16])
neumf = train_ncf_model(neumf, loader, epochs=15, lr=0.001, device=device)
neumf_metrics = eval_model(neumf, 'NeuMF')

## Compare with traditional methods

In [ ]:
# load SVD results for comparison
from src.models.mf import SVDRecommender
from src.models.popularity import PopularityRecommender

mat = build_interaction_matrix(train, N_USERS, N_ITEMS)

# popularity
pop = PopularityRecommender()
pop.fit(train)
pop_recs = pop.recommend_all(test_gt.keys(), train_user_items, n=K)
pop_m = evaluate_rankings(pop_recs, test_gt, K)

# svd
train_items_idx = {}
for _, row in train.iterrows():
    ui = int(row['user_id']) - 1
    ii = int(row['item_id']) - 1
    if ui not in train_items_idx:
        train_items_idx[ui] = set()
    train_items_idx[ui].add(ii)

svd = SVDRecommender(n_factors=50)
svd.fit(mat)
svd_recs = svd.recommend_all_users(N_USERS, train_items_idx, n=K)
svd_recs = {u: svd_recs[u] for u in test_gt if u in svd_recs}
svd_m = evaluate_rankings(svd_recs, test_gt, K)

print('\n--- Final Comparison ---')
for name, m in [('Popularity', pop_m), ('SVD', svd_m),
                ('GMF', gmf_metrics), ('MLP', mlp_metrics), ('NeuMF', neumf_metrics)]:
    print(f"{name:15s}  HR={m['HR@10']:.4f}  NDCG={m['NDCG@10']:.4f}  "
          f"Prec={m['Precision@10']:.4f}  Recall={m['Recall@10']:.4f}")